<a href="https://colab.research.google.com/github/kianajahani23-glitch/SQL/blob/main/AI_impact_on_students.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

Saving ai_student_impact_dataset (1).csv to ai_student_impact_dataset (1).csv


In [2]:
import pandas as pd

df = pd.read_csv('ai_student_impact_dataset (1).csv')
df.head()

,Student_ID,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level
0,100001,Humanities,Senior,2.418,23.31,Copywriting/Drafting,Beginner,1,True,8.13,5,Allowed_With_Citation,6,2.393,86.44,High
1,100002,Medical,Junior,3.821,1.12,Ideation,Advanced,5,False,16.65,3,Allowed_With_Citation,9,3.696,69.39,Low
2,100003,Business,Freshman,3.398,21.26,Summarizing_Reading,Beginner,2,False,10.35,5,Strict_Ban,9,3.499,73.93,Medium
3,100004,Business,Senior,3.789,1.82,Copywriting/Drafting,Intermediate,4,False,15.23,2,Allowed_With_Citation,2,4.000,63.58,Medium
4,100005,STEM,Sophomore,3.635,9.29,Debugging/Troubleshooting,Advanced,4,False,12.55,4,Allowed_With_Citation,4,3.798,100.00,Medium


In [4]:
!pip install duckdb

In [5]:
df.columns

Index(['Student_ID', 'Major_Category', 'Year_of_Study', 'Pre_Semester_GPA',
       'Weekly_GenAI_Hours', 'Primary_Use_Case', 'Prompt_Engineering_Skill',
       'Tool_Diversity', 'Paid_Subscription', 'Traditional_Study_Hours',
       'Perceived_AI_Dependency', 'Institutional_Policy',
       'Anxiety_Level_During_Exams', 'Post_Semester_GPA',
       'Skill_Retention_Score', 'Burnout_Risk_Level'],
      dtype='object')

In [6]:
import duckdb

In [8]:
duckdb.sql("""
SELECT *
FROM df
LIMIT 5
""").df()

,Student_ID,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level
0,100001,Humanities,Senior,2.418,23.31,Copywriting/Drafting,Beginner,1,True,8.13,5,Allowed_With_Citation,6,2.393,86.44,High
1,100002,Medical,Junior,3.821,1.12,Ideation,Advanced,5,False,16.65,3,Allowed_With_Citation,9,3.696,69.39,Low
2,100003,Business,Freshman,3.398,21.26,Summarizing_Reading,Beginner,2,False,10.35,5,Strict_Ban,9,3.499,73.93,Medium
3,100004,Business,Senior,3.789,1.82,Copywriting/Drafting,Intermediate,4,False,15.23,2,Allowed_With_Citation,2,4.000,63.58,Medium
4,100005,STEM,Sophomore,3.635,9.29,Debugging/Troubleshooting,Advanced,4,False,12.55,4,Allowed_With_Citation,4,3.798,100.00,Medium


In [42]:
df.dtypes

,0
Student_ID,int64
Major_Category,object
Year_of_Study,object
Pre_Semester_GPA,float64
Weekly_GenAI_Hours,float64
Primary_Use_Case,object
Prompt_Engineering_Skill,object
Tool_Diversity,int64
Paid_Subscription,bool
Traditional_Study_Hours,float64


In [9]:
duckdb.sql("""
SELECT
    Major_Category,
    AVG(Post_Semester_GPA) AS avg_gpa
FROM df
GROUP BY Major_Category
ORDER BY avg_gpa DESC
""").df()

,Major_Category,avg_gpa
0,STEM,3.363154
1,Medical,3.353168
2,Humanities,3.345825
3,Arts,3.343688
4,Business,3.336085


In [11]:
import duckdb

con = duckdb.connect()

con.register("students_df", df)

con.execute("""
CREATE TABLE students AS
SELECT *
FROM students_df
""")

In [39]:
#1.Avg Pre Semester GPA vs Avg Post Semester GPA

duckdb.sql("""
SELECT
    ROUND(AVG(Pre_Semester_GPA),2) AS Avg_Pre_GPA,
    ROUND(AVG(Post_Semester_GPA),2) AS Avg_Post_GPA
FROM df
""").df()

,Avg_Pre_GPA,Avg_Post_GPA
0,3.15,3.35


In [44]:
#2.The effect of Weekly GenAI Hours on GPA

duckdb.sql("""
SELECT
    CASE
        WHEN Weekly_GenAI_Hours < 5 THEN 'Low'
        WHEN Weekly_GenAI_Hours < 10 THEN 'Medium'
        ELSE 'High'
    END AS AI_Usage,
    COUNT(*) AS Students,
    ROUND(AVG(Post_Semester_GPA),2) AS Avg_GPA
FROM df
GROUP BY AI_Usage
ORDER BY Avg_GPA DESC
""").df()

,AI_Usage,Students,Avg_GPA
0,Medium,12182,3.37
1,High,15251,3.34
2,Low,22567,3.34


In [31]:
#3.AVG of Weekly GenAI Hours by Major Category
duckdb.sql("""
SELECT
    Major_Category,
    ROUND(AVG(Weekly_GenAI_Hours),2) AS Avg_AI_Hours
FROM df
GROUP BY Major_Category
ORDER BY Avg_AI_Hours DESC
""").df()

,Major_Category,Avg_AI_Hours
0,STEM,10.49
1,Business,8.28
2,Medical,7.55
3,Arts,7.27
4,Humanities,6.77


In [32]:
#4.Number of students in each Burnout level
duckdb.sql("""
SELECT
    Burnout_Risk_Level,
    COUNT(*) AS Student_Count
FROM df
GROUP BY Burnout_Risk_Level
ORDER BY Student_Count DESC
""").df()

,Burnout_Risk_Level,Student_Count
0,Medium,21144
1,Low,16369
2,High,12487


In [33]:
#5.Paid subscription vs Free subscription
duckdb.sql("""
SELECT
    Paid_Subscription,
    ROUND(AVG(Post_Semester_GPA),2) AS Avg_GPA,
    ROUND(AVG(Skill_Retention_Score),2) AS Avg_Retention
FROM df
GROUP BY Paid_Subscription
""").df()

,Paid_Subscription,Avg_GPA,Avg_Retention
0,False,3.35,76.07
1,True,3.35,75.42


In [34]:
#6.Top GPA Changes
duckdb.sql("""
SELECT
    Student_ID,
    ROUND(Post_Semester_GPA - Pre_Semester_GPA,2) AS GPA_Change
FROM df
ORDER BY GPA_Change DESC
LIMIT 10
""").df()

,Student_ID,GPA_Change
0,111236,1.01
1,108125,0.99
2,102060,0.94
3,128333,0.93
4,109581,0.92
5,113808,0.90
6,113509,0.90
7,102987,0.89
8,136453,0.87
9,123451,0.86


In [35]:
#7.Prompt engineerig skills by Avg GPA
duckdb.sql("""
SELECT
    Prompt_Engineering_Skill,
    ROUND(AVG(Post_Semester_GPA),2) AS Avg_GPA
FROM df
GROUP BY Prompt_Engineering_Skill
ORDER BY Prompt_Engineering_Skill
""").df()

,Prompt_Engineering_Skill,Avg_GPA
0,Advanced,3.39
1,Beginner,3.33
2,Intermediate,3.33


In [36]:
#8.Average retention vs Ai dependency
duckdb.sql("""
SELECT
    Perceived_AI_Dependency,
    ROUND(AVG(Skill_Retention_Score),2) AS Avg_Retention
FROM df
GROUP BY Perceived_AI_Dependency
ORDER BY Avg_Retention DESC
""").df()

,Perceived_AI_Dependency,Avg_Retention
0,3,76.48
1,2,76.48
2,4,76.35
3,1,76.00
4,5,75.85
5,6,74.90
6,7,72.47
7,8,69.40
8,9,65.56
9,10,63.55


In [40]:
#9.Average GPAs by Major categories
duckdb.sql("""
SELECT
    Major_Category,
    ROUND(AVG(Post_Semester_GPA),2) AS Avg_GPA,
    RANK() OVER (
        ORDER BY AVG(Post_Semester_GPA) DESC
    ) AS GPA_Rank
FROM df
GROUP BY Major_Category
""").df()

,Major_Category,Avg_GPA,GPA_Rank
0,STEM,3.36,1
1,Medical,3.35,2
2,Humanities,3.35,3
3,Arts,3.34,4
4,Business,3.34,5


In [41]:
#Intitutional Policies
duckdb.sql("""
SELECT
    Institutional_Policy,
    COUNT(*) AS Students,
    ROUND(AVG(Post_Semester_GPA),2) AS Avg_GPA
FROM df
GROUP BY Institutional_Policy
ORDER BY Avg_GPA DESC
""").df()

,Institutional_Policy,Students,Avg_GPA
0,Allowed_With_Citation,25224,3.35
1,Actively_Encouraged,14988,3.35
2,Strict_Ban,9788,3.33
